# Experiment 1 · Notebook 05: Feature inspection and the OOD probe

*J26-IT-304 PP1. Baseline confusion classifier (TF-IDF + Logistic Regression). Sections 10-11.*

Looks inside the final model (which features it uses, whether they are stable, whether it is "just length") and
then evaluates it on the frozen out-of-distribution probe, which measures the gap between the correctness proxy and
the confusion the deployed tutor must detect. **The model is final here:** nothing in this notebook changes its
hyperparameters, training data, threshold or test metrics.

Section numbers run continuously across the Experiment 1 series, so a reference such as "EDA §3.5" or "section 8" points to the same place in every notebook:

| Notebook | Sections | Stage |
|---|---|---|
| `01_data_and_eda` | §1-§3 | provenance, load, exploratory analysis, leakage demonstration |
| `02_cleaning_and_labels` | §4-§5 | deduplication, group key, label mapping, proxy caveat |
| `03_split_and_baselines` | §6-§7 | group-aware split, trivial baselines, frozen OOD probe |
| `04_model_training_and_evaluation` | §8-§9 | TF-IDF + LR, group-aware CV, the single test evaluation |
| `05_features_and_ood_probe` | §10-§11 | coefficient inspection, stability, length, OOD probe evaluation |

Each notebook runs on its own from a clean kernel (kernel `exp1-confusion`). Its first code cell rebuilds its inputs with the same `confusion_baseline` functions the previous stage used, so the stages cannot drift apart. Run them in order 01 → 05 to regenerate every file in `reports/experiments/`.

**Setup.** Imports, and the inputs of this stage: the split of notebook 03, rebuilt and **proved identical** by its
fingerprint; the final model, refitted from the hyperparameters notebook 04 saved (`chosen_params_primary.json`); and the
length-only baseline. A **reproduction check** then re-scores the test set once and asserts the numbers equal
notebook 04's saved comparison table, which proves this is the same frozen model 04 evaluated. No decision depends on
that scoring, and no other cell predicts on the test set.

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from confusion_baseline import REPORTS_DIR, SEED
from confusion_baseline.data import load_raw
from confusion_baseline.evaluate import metrics
from confusion_baseline.model import build_pipeline, cv_folds, length_baseline, load_params, token_count
from confusion_baseline.prep import clean, map_labels
from confusion_baseline.probe import load_probe, probe_sha256
from confusion_baseline.split import group_split, split_fingerprint

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"font.size": 11})
FIG_DIR = REPORTS_DIR / "experiments"

# ---- inputs: the same split as notebooks 03 / 04 ----
df, followup_quarantine, _ = load_raw()
del followup_quarantine
y_primary = map_labels(clean(df)[0], "primary")
train, test = group_split(y_primary, test_size=0.2, seed=SEED)
U4_FINGERPRINT = "511fadfecbff1c84dad9f22c4ecdd51052b13c6c1187c6f60d2468342905e44c"
fp = split_fingerprint(test)
assert fp == U4_FINGERPRINT, fp
assert not set(train["group_key"]) & set(test["group_key"])
print(f"PASS: same test set as notebooks 03/04 (fingerprint {fp[:12]}...); train {len(train)} rows, test {len(test)} rows, 0 shared questions")

# ---- the final model: chosen params from notebook 04, refitted on train ----
params = load_params(FIG_DIR / "chosen_params_primary.json")
final = build_pipeline(**params).fit(train["Answer"], train["y"])
len_clf = length_baseline(train)
print("params:", params)

# ---- reproduction check: re-score test ONCE and compare with notebook 04's saved table ----
pred_model = final.predict(test["Answer"])
proba_model = final.predict_proba(test["Answer"])[:, 1]
pred_len = len_clf.predict(token_count(test["Answer"]))
y_te = test["y"].to_numpy()
saved = pd.read_csv(FIG_DIR / "comparison_primary.csv").set_index("System")
cols = {"accuracy": "Accuracy", "precision_confused": "Precision", "recall_confused": "Recall",
        "f1_confused": "F1 (confused)", "macro_f1": "Macro F1"}
for name, pred in [("TF-IDF + LR", pred_model), ("Length-only LR", pred_len)]:
    got = {cols[k]: round(v, 4) for k, v in metrics(y_te, pred).items()}
    want = {c: round(float(saved.loc[name, c]), 4) for c in cols.values()}
    assert got == want, (name, got, want)
    print(f"PASS: {name} reproduces notebook 04's row exactly: F1(confused) {got['F1 (confused)']}, macro F1 {got['Macro F1']}")

PASS: same test set as notebooks 03/04 (fingerprint 511fadfecbff...); train 860 rows, test 216 rows, 0 shared questions
params: {'clf__C': 100, 'clf__class_weight': 'balanced', 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__stop_words': None, 'tfidf__sublinear_tf': True}
PASS: TF-IDF + LR reproduces notebook 04's row exactly: F1(confused) 0.7581, macro F1 0.6629
PASS: Length-only LR reproduces notebook 04's row exactly: F1(confused) 0.7651, macro F1 0.6214


## 10. Feature inspection

**Why.** Interpretable coefficients are the reason a lexical model was chosen (ARCHITECTURE ADR-03). They answer the
examiner's question "which features is it using?", and they are item 6 of the spec's leakage checklist: are the
learned features *semantic*, or *generator artefacts*? Positive coefficients push a prediction toward **confused**
(y = 1), negative ones toward **understood**. The model was fitted with `C=100` (weak regularisation) and reaches a
training macro F1 of 1.000, so individual coefficients may be noise; 10.3 tests that.

### 10.1 Top coefficients

In [2]:
vec, clf = final.named_steps["tfidf"], final.named_steps["clf"]
assert list(clf.classes_) == [0, 1]                       # coefficient > 0  =>  toward y = 1 = confused
terms = vec.get_feature_names_out()
coef = clf.coef_[0]
print("vocabulary size:", len(terms), "| terms with a non-zero coefficient:", int((coef != 0).sum()))

present = (vec.transform(train["Answer"]) > 0).tocsc()      # document presence in TRAIN (vectoriser already fitted on train)
is_conf = train["y"].to_numpy() == 1
n_conf, n_und = int(is_conf.sum()), int((~is_conf).sum())
df_conf = np.asarray(present[is_conf].sum(axis=0)).ravel()
df_und = np.asarray(present[~is_conf].sum(axis=0)).ravel()

def top_table(order, direction):
    return pd.DataFrame({
        "direction": direction, "rank": range(1, len(order) + 1), "term": terms[order], "coef": coef[order].round(3),
        "df_understood": df_und[order], "df_confused": df_conf[order],
        "share_understood": (df_und[order] / n_und).round(3), "share_confused": (df_conf[order] / n_conf).round(3)})

order = np.argsort(coef, kind="stable")
pos = top_table(order[::-1][:20], "-> confused")
neg = top_table(order[:20], "-> understood")
print(f"\nTOP 20 POSITIVE coefficients (toward CONFUSED). df = number of TRAIN answers containing the term "
      f"({n_und} understood, {n_conf} confused answers in train):")
display(pos.drop(columns="direction").set_index("rank"))
print("TOP 20 NEGATIVE coefficients (toward UNDERSTOOD):")
display(neg.drop(columns="direction").set_index("rank"))

top40 = pd.concat([pos, neg], ignore_index=True)
top40.to_csv(FIG_DIR / "top_features_primary.csv", index=False)
print("saved: reports/experiments/top_features_primary.csv", top40.shape)

vocabulary size: 8548 | terms with a non-zero coefficient: 8548

TOP 20 POSITIVE coefficients (toward CONFUSED). df = number of TRAIN answers containing the term (333 understood, 527 confused answers in train):


,term,coef,df_understood,df_confused,share_understood,share_confused
rank,,,,,,
1,rounds the,6.542,0,4,0.000,0.008
2,makes,5.503,1,16,0.003,0.030
3,program,5.355,18,54,0.054,0.102
4,returns an,5.060,1,4,0.003,0.008
5,used to,4.923,32,75,0.096,0.142
6,error,4.882,3,20,0.009,0.038
7,but it,4.710,5,30,0.015,0.057
8,log,4.592,0,2,0.000,0.004
9,is log,4.592,0,2,0.000,0.004


TOP 20 NEGATIVE coefficients (toward UNDERSTOOD):


,term,coef,df_understood,df_confused,share_understood,share_confused
rank,,,,,,
1,of,-6.564,141,140,0.423,0.266
2,implementation,-5.927,12,2,0.036,0.004
3,to,-5.848,181,231,0.544,0.438
4,to the,-5.137,44,38,0.132,0.072
5,which,-5.115,35,26,0.105,0.049
6,or,-5.000,53,48,0.159,0.091
7,double,-4.816,13,8,0.039,0.015
8,and then,-4.739,5,1,0.015,0.002
9,method of,-4.514,4,0,0.012,0.000


saved: reports/experiments/top_features_primary.csv (40, 8)


### 10.2 What kind of feature is each term?

**Why.** Reading the 40 terms one by one is the only way to judge them. Each is assigned to exactly one category,
with a one-line reason: **semantic_wrongness** (plausibly signals an incorrect or incomplete answer),
**semantic_correctness** (precise technical content typical of complete answers), **length_proxy** (function words or
connectives that mainly mark longer, fuller answers), **topic** (names a Java topic: identifies *what is asked*, not
whether it is right) or **artefact** (template fragments, odd tokens, generator style). The assignment below was written
by hand after reading the terms and the document-frequency columns above; the cell asserts that every one of the 40
terms is covered exactly once.

In [3]:
CATS = {
    # ---- toward CONFUSED ----
    "rounds the": ("topic", "ceil()/floor()/cast question family; all 4 train answers containing it are confused"),
    "makes": ("semantic_wrongness", "vague causal claim ('final makes sure the code is more secure'); 16 of 17 answers are confused"),
    "program": ("artefact", "template opening 'The program will ...', the most common opening of Wrong answers (EDA 3.5)"),
    "returns an": ("topic", "return-type questions (ceil(), valueOf()); 5 answers only"),
    "used to": ("semantic_wrongness", "generic 'X is used to ...' definition without the precise detail"),
    "error": ("semantic_wrongness", "wrong answers claim errors that do not occur ('will throw an error')"),
    "but it": ("semantic_wrongness", "qualifier clause typical of partial answers ('..., but it can only ...')"),
    "log": ("topic", "complexity questions: the O(log n) claim; only 2 train answers, so rare"),
    "is log": ("topic", "same two answers as 'log' (O(log n))"),
    "but": ("semantic_wrongness", "contrast clause marking a qualified, partial answer"),
    "only": ("semantic_wrongness", "absolute 'can only be used / only divisible by', typical of over-restricted claims"),
    "used": ("semantic_wrongness", "generic 'is used for' phrasing, vague rather than precise"),
    "the program": ("artefact", "same 'The program will ...' generator template as 'program'"),
    "an error": ("semantic_wrongness", "claims an error where there is none; 13 of 14 answers are confused"),
    "doesn": ("artefact", "tokeniser fragment of \"doesn't\" (sklearn splits at the apostrophe); negated claims"),
    "the int": ("topic", "int / wrapper-type questions; 2 train answers, so rare"),
    "to execute": ("topic", "JVM / JRE / polymorphism-at-runtime answers; all 7 are confused"),
    "them": ("topic", "encapsulation answers ('keep variables private ... access them'); 76% confused"),
    "that is": ("artefact", "generic definitional connective ('a special method that is called ...') with no clear meaning"),
    "greater": ("topic", "numeric-comparison answers (prime numbers, wrapper 'greater than 0'); all 7 are confused"),
    # ---- toward UNDERSTOOD ----
    "of": ("length_proxy", "function word; understood answers use more noun phrases (42% vs 27% of answers contain it)"),
    "implementation": ("semantic_correctness", "precise technical term (interface vs abstract class); 12 understood vs 2 confused"),
    "to": ("length_proxy", "function word marking fuller sentences (54% vs 44%)"),
    "to the": ("length_proxy", "function-word bigram"),
    "which": ("length_proxy", "relative-clause connective that builds longer answers"),
    "or": ("length_proxy", "coordinating connective, fuller answers"),
    "double": ("topic", "double / cast questions"),
    "and then": ("length_proxy", "sequencing connective typical of longer explanations; 5 vs 1"),
    "method of": ("semantic_correctness", "precise relation ('method of the superclass / child class'); 4 understood, 0 confused"),
    "development": ("topic", "JDK vs JRE questions ('for development')"),
    "actual": ("semantic_correctness", "wording such as 'the actual object / runtime type'; 9 vs 2"),
    "defined": ("semantic_correctness", "precise 'no constructor is defined in the class'; 9 vs 1"),
    "of equals": ("topic", "equals() return-type family; 1 vs 1, so effectively noise"),
    "typically": ("semantic_correctness", "precisely qualified statement ('typically used when the number of iterations is known'); 10 vs 2"),
    "whereas": ("length_proxy", "contrast connective that builds longer comparisons"),
    "if": ("length_proxy", "conditional connective, fuller answers (19% vs 15%)"),
    "the actual": ("semantic_correctness", "same 'actual (runtime) type' wording"),
    "declared": ("semantic_correctness", "precise 'declared final' / 'handled or declared'; 7 vs 2"),
    "the integer": ("topic", "Integer / parseInt question family; 5 understood, 0 confused"),
    "native machine": ("semantic_correctness", "complete JIT description ('compiling bytecode into native machine code'); 5 vs 0"),
}
assert set(CATS) == set(top40["term"]), set(CATS) ^ set(top40["term"])
assert len(CATS) == 40 and top40["term"].is_unique
top40["category"] = top40["term"].map(lambda t: CATS[t][0])
top40["reason"] = top40["term"].map(lambda t: CATS[t][1])
print("every one of the 40 terms is categorised exactly once")
display(top40[["direction", "rank", "term", "coef", "category", "reason"]].set_index(["direction", "rank"]))
cat_counts = pd.crosstab(top40["category"], top40["direction"], margins=True)
print("category counts:")
display(cat_counts)
top40.to_csv(FIG_DIR / "top_features_primary.csv", index=False)

every one of the 40 terms is categorised exactly once


term   coef              category                                                                                     reason
direction     rank                                                                                                                                        
-> confused   1         rounds the  6.542                 topic        ceil()/floor()/cast question family; all 4 train answers containing it are confused
              2              makes  5.503    semantic_wrongness  vague causal claim ('final makes sure the code is more secure'); 16 of 17 answers are ...
              3            program  5.355              artefact  template opening 'The program will ...', the most common opening of Wrong answers (EDA...
              4         returns an  5.060                 topic                                  return-type questions (ceil(), valueOf()); 5 answers only
              5            used to  4.923    semantic_wrongness                           generic 'X is used to ...' definition without the precise detail
              6              error  4.882    semantic_wrongness                       wrong answers claim errors that do not occur ('will throw an error')
              7             but it  4.710    semantic_wrongness                   qualifier clause typical of partial answers ('..., but it can only ...')
              8                log  4.592                 topic                    complexity questions: the O(log n) claim; only 2 train answers, so rare
              9             is log  4.592                 topic                                                       same two answers as 'log' (O(log n))
              10               but  4.575    semantic_wrongness                                        contrast clause marking a qualified, partial answer
              11              only  4.572    semantic_wrongness         absolute 'can only be used / only divisible by', typical of over-restricted claims
              12              used  4.522    semantic_wrongness                                  generic 'is used for' phrasing, vague rather than precise
              13       the program  4.504              artefact                                same 'The program will ...' generator template as 'program'
              14          an error  3.917    semantic_wrongness                         claims an error where there is none; 13 of 14 answers are confused
              15             doesn  3.874              artefact         tokeniser fragment of "doesn't" (sklearn splits at the apostrophe); negated claims
              16           the int  3.617                 topic                                     int / wrapper-type questions; 2 train answers, so rare
              17        to execute  3.597                 topic                            JVM / JRE / polymorphism-at-runtime answers; all 7 are confused
              18              them  3.542                 topic             encapsulation answers ('keep variables private ... access them'); 76% confused
              19           that is  3.498              artefact  generic definitional connective ('a special method that is called ...') with no clear ...
              20           greater  3.495                 topic   numeric-comparison answers (prime numbers, wrapper 'greater than 0'); all 7 are confused
-> understood 1                 of -6.564          length_proxy  function word; understood answers use more noun phrases (42% vs 27% of answers contain...
              2     implementation -5.927  semantic_correctness          precise technical term (interface vs abstract class); 12 understood vs 2 confused
              3                 to -5.848          length_proxy                                        function word marking fuller sentences (54% vs 44%)
              4             to the -5.137          length_proxy                                                                       function-word bigram


category counts:


direction,-> confused,-> understood,All
category,,,
artefact,4,0,4
length_proxy,0,8,8
semantic_correctness,0,8,8
semantic_wrongness,8,0,8
topic,8,4,12
All,20,20,40


**Verdict for leakage-checklist item 6: mostly semantic or structural, not a generator leak, with two caveats.** The 40
terms split into three groups (counts above):

- **Plausibly semantic (16 of 40: 8 wrongness, 8 correctness).** On the confused side: absolutes and qualifiers (`only`, `but`, `but it`),
  vague phrasing (`used`, `used to`, `makes`) and claims of errors that do not occur (`error`, `an error`). On the understood
  side: precise technical wording (`implementation`, `defined`, `declared`, `typically`, `actual`, `native machine`). This is what
  a correctness proxy *should* pick up.
- **Length / function words (8, all on the understood side).** `of`, `to`, `to the`, `which`, `or`, `and then`, `whereas`, `if`
  mark fuller sentences. So even though TF-IDF is length-normalised, the model recovers part of the length signal through
  these words, which fits the length-only baseline scoring about the same.
- **Topic (12) and artefact (4) terms.** Topic terms (`rounds the`, `greater`, `to execute`, `double`, `the integer`, ...) name
  *what is asked*. Test questions are unseen, so they can only help through question families (floor()/ceil()/cast, JDK/JRE), and
  several rest on 2-7 training answers. Artefact terms are the generator template `program` / `the program` ("The program will
  ...", the common Wrong opening) and tokeniser fragments (`doesn`, `that is`).

No term is a leak of the kind the checklist worries about (FollowUp text, question identifiers, the U+2019 character). But the lexical signal is **generator style as much as correctness**: this is what a
synthetic dataset gives, and it is why the probe in section 11 matters.

### 10.3 Stability

**Why.** With `C=100` and a perfect training fit, a term can sit in the top 20 because of a handful of training
answers. A simple stability test: refit the same pipeline on the training part of each of the 5 group-aware CV folds
(`cv_folds(train)`, **train only**, no test data) and count, for each of the 40 final top terms, in how many of the 5
fold models it is also in the top 20 *in the same direction*. Terms in 0-1 folds are treated as noise.

In [4]:
folds = cv_folds(train)
fold_top = []
for k, (tr_idx, _) in enumerate(folds, 1):
    m_k = build_pipeline(**params).fit(train["Answer"].iloc[tr_idx], train["y"].iloc[tr_idx])
    t_k = m_k.named_steps["tfidf"].get_feature_names_out()
    c_k = m_k.named_steps["clf"].coef_[0]
    o_k = np.argsort(c_k, kind="stable")
    fold_top.append({"confused": set(t_k[o_k[::-1][:20]]), "understood": set(t_k[o_k[:20]])})

def folds_present(row):
    side = "confused" if row["direction"] == "-> confused" else "understood"
    return sum(row["term"] in ft[side] for ft in fold_top)

top40["folds_in_top20 (of 5)"] = top40.apply(folds_present, axis=1)
stab = top40[["direction", "rank", "term", "coef", "folds_in_top20 (of 5)"]]
print("stability of the 40 final top terms across the 5 train-only fold models:")
display(stab.set_index(["direction", "rank"]))

counts = top40["folds_in_top20 (of 5)"].value_counts().reindex(range(6), fill_value=0)
print("terms by number of folds in which they stay in the top 20 (same direction):")
print(counts.rename_axis("folds").to_frame("n terms").T.to_string())
n_noise = int((top40["folds_in_top20 (of 5)"] <= 1).sum())
n_stable = int((top40["folds_in_top20 (of 5)"] >= 4).sum())
print(f"noise (0-1 folds): {n_noise} of 40 | moderately stable (2-3): {40 - n_noise - n_stable} | stable (4-5): {n_stable}")
for side in ["-> confused", "-> understood"]:
    sub = top40[top40["direction"] == side]["folds_in_top20 (of 5)"]
    print(f"  {side:14s} mean folds present: {sub.mean():.2f}; noise {int((sub <= 1).sum())}/20; stable {int((sub >= 4).sum())}/20")

stability of the 40 final top terms across the 5 train-only fold models:


term   coef  folds_in_top20 (of 5)
direction     rank                                              
-> confused   1         rounds the  6.542                      4
              2              makes  5.503                      4
              3            program  5.355                      5
              4         returns an  5.060                      5
              5            used to  4.923                      5
              6              error  4.882                      5
              7             but it  4.710                      4
              8                log  4.592                      4
              9             is log  4.592                      4
              10               but  4.575                      5
              11              only  4.572                      5
              12              used  4.522                      5
              13       the program  4.504                      5
              14          an error  3.917                      3
              15             doesn  3.874                      2
              16           the int  3.617                      2
              17        to execute  3.597                      2
              18              them  3.542                      2
              19           that is  3.498                      1
              20           greater  3.495                      2
-> understood 1                 of -6.564                      5
              2     implementation -5.927                      5
              3                 to -5.848                      5
              4             to the -5.137                      5
              5              which -5.115                      4
              6                 or -5.000                      4
              7             double -4.816                      4
              8           and then -4.739                      4
              9          method of -4.514                      4
              10       development -4.462                      2
              11            actual -4.460                      4
              12           defined -4.446                      3
              13         of equals -4.357                      2
              14         typically -4.354                      3
              15           whereas -4.219                      3
              16                if -4.170                      3
              17        the actual -4.149                      4
              18          declared -4.014                      3
              19       the integer -4.012                      3
              20    native machine -3.960                      1

terms by number of folds in which they stay in the top 20 (same direction):
folds    0  1  2  3   4   5
n terms  0  2  7  7  12  12
noise (0-1 folds): 2 of 40 | moderately stable (2-3): 14 | stable (4-5): 24
  -> confused    mean folds present: 3.70; noise 1/20; stable 13/20
  -> understood  mean folds present: 3.55; noise 1/20; stable 11/20


**Reading the stability.** The list is **mostly stable**: 24 of the 40 terms stay in the top 20 in 4 or 5 of the 5 train-only
fold models, 14 in 2-3, and only 2 in 0-1 fold (noise). Both directions behave alike (mean 3.7 and 3.55 folds). The
unstable ones are the rare terms (`the int`, `to execute`, `greater`, `native machine`, `development`, `of equals`), each resting on
2-7 answers, which is exactly where `C=100` lets a few examples dominate. The frequent terms (`program`, `used`, `but`, `only`,
`of`, `to`, `which`, `or`) are the stable core. So the picture in 10.2 holds for the stable core, and the rare topic terms
should not be quoted as findings.

### 10.4 Is it just length?

**Why.** Length-only logistic regression scores about the same as TF-IDF + LR on the test set (notebook 04), yet TF-IDF
vectors are length-normalised. Two checks: how strongly the model's score tracks answer length, and whether the two
systems get the *same* rows right. If they score alike but disagree on many rows, they use different signals.

In [5]:
tok_len = test["Answer"].str.split().str.len().to_numpy()
r = float(np.corrcoef(proba_model, tok_len)[0, 1])
print(f"corr(P(confused), token length) on test = {r:+.4f}    (equivalently corr with -length = {-r:+.4f})")
print(f"corr(length-only LR prediction, token length) on test = {float(np.corrcoef(pred_len, tok_len)[0, 1]):+.4f}")

ok_m, ok_l = pred_model == y_te, pred_len == y_te
agree = pd.Series({
    "both right": int((ok_m & ok_l).sum()),
    "only TF-IDF + LR right": int((ok_m & ~ok_l).sum()),
    "only length-only right": int((~ok_m & ok_l).sum()),
    "both wrong": int((~ok_m & ~ok_l).sum()),
}, name="rows")
display(agree.to_frame())
print("total:", int(agree.sum()), "| the two systems give a different prediction on",
      int((pred_model != pred_len).sum()), "of", len(test), "rows")
print(f"accuracy: TF-IDF + LR {ok_m.mean():.4f} | length-only {ok_l.mean():.4f}")

corr(P(confused), token length) on test = -0.1538    (equivalently corr with -length = +0.1538)
corr(length-only LR prediction, token length) on test = -0.7285


,rows
both right,106
only TF-IDF + LR right,43
only length-only right,40
both wrong,27


total: 216 | the two systems give a different prediction on 83 of 216 rows
accuracy: TF-IDF + LR 0.6898 | length-only 0.6759


**Reading 10.4.** The model's score barely tracks length: corr(P(confused), token length) = **-0.154** on test (a longer answer
is slightly less likely to be called confused), far weaker than the length-only rule, which is a threshold on length
(corr -0.73 with length). Yet the two systems score about the same (accuracy 0.690 vs 0.676). They **disagree on 83 of 216
rows (38%)**: both are right on 106 rows, only TF-IDF + LR on 43, only length-only on 40, and both wrong on 27. So they get
their similar scores from **different signals**: the lexical model from vocabulary (plus the length-correlated function words
seen in 10.2), the baseline from word count alone. Because their errors overlap only partly, a model that combines them
might do better than either, but **that is a PP2 question and is not built here**.

### 10.5 The U+2019 artefact

**Why.** EDA section 3.5 and notebook 02 (section 4) found that the curly apostrophe (U+2019) is about twice as frequent in
confused answers, a generator artefact. Word-level TF-IDF should be unable to use it. This confirms it on the fitted vocabulary.

In [6]:
nonascii = [t for t in terms if not t.isascii()]
print("vocabulary terms containing a non-ASCII character:", len(nonascii), nonascii[:10])
print("vocabulary terms containing an ASCII apostrophe (for comparison):", len([t for t in terms if "'" in t]))
assert len(nonascii) == 0
print("PASS: no vocabulary term contains a non-ASCII character, so the model cannot use U+2019")

vocabulary terms containing a non-ASCII character: 0 []
vocabulary terms containing an ASCII apostrophe (for comparison): 0
PASS: no vocabulary term contains a non-ASCII character, so the model cannot use U+2019


**Reading 10.5.** The fitted vocabulary has **0** terms with a non-ASCII character (and none with an apostrophe at all: the
default token pattern splits `doesn't` into `doesn` and drops the apostrophe). The U+2019 artefact therefore cannot reach the
model, as the earlier word-level token check predicted.

## 11. The OOD probe

**Why.** The deployed tutor will mostly receive messages like "wait, I don't follow", which are confused but contain no
answer to mark, and the training data has none (0 hedging words in 1,078 answers). The probe (`probe/ood_probe.yaml`, 40 hand-written
items, **frozen at U4** before any model existed) measures how far the correctness proxy carries over. It is
**evaluation only**: it is never trained on, never used to tune anything, and never averaged into the test metrics.
It is expected to score badly; that quantifies the proxy gap and is the evidence for the rule layer.

**How to read it.** The probe is 3 : 1 towards confused (30 expected-confused, 10 expected-understood), so a predictor that
says "confused" for everything gets **accuracy 0.75** on it. Accuracy alone is therefore never reported: the results are given
**per category** (recall on each confused category, specificity on `explicit_understanding`) next to that constant predictor
and the length-only baseline.

In [7]:
probe = load_probe()
print("probe items:", len(probe), "| file sha256:", probe_sha256())
assert probe_sha256().startswith("cbccf0a0"), "probe file changed since U4"
print("PASS: probe file is the one frozen at U4 (sha256 cbccf0a0...)")

probe["p_confused"] = final.predict_proba(probe["text"])[:, 1]
probe["pred_model"] = final.predict(probe["text"])
probe["pred_length"] = len_clf.predict(token_count(probe["text"]))
probe["tokens"] = probe["text"].str.split().str.len()
probe["ok_model"] = probe["pred_model"] == probe["y"]
probe["ok_length"] = probe["pred_length"] == probe["y"]

# vocabulary coverage: share of unigram tokens the fitted vocabulary knows (unknown words give no signal)
from sklearn.feature_extraction.text import TfidfVectorizer
unigram = TfidfVectorizer(ngram_range=(1, 1)).build_analyzer()
vocab = set(vec.vocabulary_)
known = probe["text"].map(lambda t: [w for w in unigram(t) if w in vocab])
probe["known_unigrams"] = known.map(len)
probe["coverage"] = (known.map(len) / probe["text"].map(lambda t: max(len(unigram(t)), 1))).round(2)
print("items with ZERO in-vocabulary unigrams (prediction = intercept only):", int((probe["known_unigrams"] == 0).sum()),
      "| intercept =", round(float(clf.intercept_[0]), 3), "-> P(confused) =", round(float(1 / (1 + np.exp(-clf.intercept_[0]))), 3))
print(f"mean coverage {probe['coverage'].mean():.2f} | probe items are short: median {probe['tokens'].median():.0f} tokens "
      f"(training answers: median {train['Answer'].str.split().str.len().median():.0f})")

probe items: 40 | file sha256: cbccf0a0c0d10eded7ac31df41d4af1afde49c66242f685a6cfea41e1e9908f5
PASS: probe file is the one frozen at U4 (sha256 cbccf0a0...)
items with ZERO in-vocabulary unigrams (prediction = intercept only): 1 | intercept = 1.451 -> P(confused) = 0.81
mean coverage 0.71 | probe items are short: median 7 tokens (training answers: median 19)


**Per-item results.** Every item with its expected label, the model's P(confused) and the predictions of the model and the
length-only baseline (0 = understood, 1 = confused).

In [8]:
view = probe[["id", "category", "expected", "text", "p_confused", "pred_model", "pred_length", "coverage"]].copy()
view["p_confused"] = view["p_confused"].round(3)
view["model"] = np.where(probe["ok_model"], "ok", "WRONG")
view["length"] = np.where(probe["ok_length"], "ok", "WRONG")
for cat in ["explicit_confusion", "topic_confusion", "hedged_attempt", "explicit_understanding"]:
    print(f"--- {cat} ---")
    print(view[view["category"] == cat].drop(columns="category").to_string(index=False))

--- explicit_confusion ---
 id expected                              text  p_confused  pred_model  pred_length  coverage model length
e01 confused                I don't understand       0.942           1            1      0.50    ok     ok
e02 confused                          I’m lost       0.810           1            1      0.00    ok     ok
e03 confused                           no idea       0.923           1            1      0.50    ok     ok
e04 confused        can you explain that again       0.681           1            1      0.80    ok     ok
e05 confused                  what do you mean       0.850           1            1      0.75    ok     ok
e06 confused              wait, I don’t follow       0.726           1            1      1.00    ok     ok
e07 confused         this makes no sense to me       0.993           1            1      0.67    ok     ok
e08 confused          im so confused right now       0.938           1            1      0.60    ok     ok
e09 confus

**Per category and overall.** For a confused category the relevant number is *recall* (share flagged confused); for
`explicit_understanding` it is *specificity* (share correctly left as understood). Overall metrics treat confused as the
positive class, as everywhere else, with the constant "always confused" predictor and the length-only baseline beside the model.

In [9]:
rows = []
for cat in ["explicit_confusion", "topic_confusion", "hedged_attempt", "explicit_understanding"]:
    s = probe[probe["category"] == cat]
    rows.append({"category": cat, "expected": s["expected"].iloc[0], "n": len(s),
                 "model correct": int(s["ok_model"].sum()), "model rate": round(float(s["ok_model"].mean()), 3),
                 "length-only correct": int(s["ok_length"].sum()), "length-only rate": round(float(s["ok_length"].mean()), 3),
                 "mean P(confused)": round(float(s["p_confused"].mean()), 3),
                 "rate means": "recall" if s["expected"].iloc[0] == "confused" else "specificity"})
per_cat = pd.DataFrame(rows)
display(per_cat)

y_p = probe["y"].to_numpy()
systems = {"Always confused": np.ones(len(probe), dtype=int), "Length-only LR": probe["pred_length"].to_numpy(),
           "TF-IDF + LR": probe["pred_model"].to_numpy()}
overall = []
for name, pr in systems.items():
    mm = metrics(y_p, pr)
    spec = float(((pr == 0) & (y_p == 0)).sum() / (y_p == 0).sum())
    overall.append({"System": name, "Accuracy": mm["accuracy"], "Precision": mm["precision_confused"],
                    "Recall (confused)": mm["recall_confused"], "Specificity": spec,
                    "Balanced acc.": (mm["recall_confused"] + spec) / 2, "F1 (confused)": mm["f1_confused"], "Macro F1": mm["macro_f1"]})
overall = pd.DataFrame(overall).round(4)
display(overall)

from sklearn.metrics import roc_auc_score

def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

print(f"ROC AUC of P(confused) on the probe (ranking only, no threshold): {roc_auc_score(y_p, probe['p_confused']):.4f}   (0.5 = chance)")
for label, k, n in [("recall on the 30 expected-confused items", int(probe.loc[probe.y == 1, "ok_model"].sum()), 30),
                    ("specificity on the 10 expected-understood items", int(probe.loc[probe.y == 0, "ok_model"].sum()), 10)]:
    lo, hi = wilson(k, n)
    print(f"model {label}: {k}/{n} = {k / n:.3f}, 95% Wilson interval [{lo:.3f}, {hi:.3f}]")
print("share of probe items the model flags as confused:", int(probe["pred_model"].sum()), "of", len(probe),
      "| the length-only baseline flags:", int(probe["pred_length"].sum()), "of", len(probe))

cm = pd.crosstab(probe["expected"], probe["pred_model"].map({0: "pred understood", 1: "pred confused"}))
print("TF-IDF + LR confusion on the probe (rows = expected):")
display(cm)

per_cat.to_csv(FIG_DIR / "probe_per_category_primary.csv", index=False)
overall.to_csv(FIG_DIR / "probe_overall_primary.csv", index=False)
probe.drop(columns=["y"]).round(4).to_csv(FIG_DIR / "probe_items_primary.csv", index=False)
print("saved: reports/experiments/probe_per_category_primary.csv, probe_overall_primary.csv, probe_items_primary.csv")

,category,expected,n,model correct,model rate,length-only correct,length-only rate,mean P(confused),rate means
0,explicit_confusion,confused,10,9,0.9,10,1.0,0.823,recall
1,topic_confusion,confused,10,9,0.9,10,1.0,0.757,recall
2,hedged_attempt,confused,10,8,0.8,10,1.0,0.829,recall
3,explicit_understanding,understood,10,2,0.2,0,0.0,0.694,specificity


,System,Accuracy,Precision,Recall (confused),Specificity,Balanced acc.,F1 (confused),Macro F1
0,Always confused,0.75,0.7500,1.0000,0.0,0.5000,0.8571,0.4286
1,Length-only LR,0.75,0.7500,1.0000,0.0,0.5000,0.8571,0.4286
2,TF-IDF + LR,0.70,0.7647,0.8667,0.2,0.5333,0.8125,0.5312


ROC AUC of P(confused) on the probe (ranking only, no threshold): 0.5733   (0.5 = chance)
model recall on the 30 expected-confused items: 26/30 = 0.867, 95% Wilson interval [0.703, 0.947]
model specificity on the 10 expected-understood items: 2/10 = 0.200, 95% Wilson interval [0.057, 0.510]
share of probe items the model flags as confused: 34 of 40 | the length-only baseline flags: 40 of 40
TF-IDF + LR confusion on the probe (rows = expected):


pred_model,pred confused,pred understood
expected,,
confused,26,4
understood,8,2


saved: reports/experiments/probe_per_category_primary.csv, probe_overall_primary.csv, probe_items_primary.csv


**Reading section 11: the proxy gap is large, and the model is not a confusion detector.**

- **It says "confused" to almost everything.** It flags 34 of the 40 probe items as confused. Recall on the 30 expected-confused
  items looks good (26/30 = 0.867, 95% Wilson interval 0.70 to 0.95), but **specificity on the 10 expected-understood items is
  2/10 = 0.20 (interval 0.06 to 0.51)**: it calls 8 of 10 students who say "got it" or "that makes sense" confused.
- **Against the constant predictor it adds almost nothing.** Accuracy is **0.70, below the 0.75 of "always confused"**; balanced
  accuracy is 0.533 (chance is 0.5); macro F1 is 0.531 vs 0.429 for the constant predictor; and the ROC AUC of P(confused) is
  **0.573**, close to chance, so even the *ranking* barely separates confused from understood items. The gain in macro F1 comes
  from the two understood items it gets right, not from a reliable signal.
- **Why.** (1) The probe items are short (median 7 tokens against 19 in training) and only 71% of their words are in the model's
  vocabulary; one item (`e02`, "I'm lost") has no known word at all and gets the intercept-only prediction, P(confused) = 0.81. The model's
  default for text it does not recognise is "confused". (2) Terms learned from generator style hit real student phrases: `makes`, the
  second strongest confused term (10.1), pushes "that makes sense" to P = 0.992 confused. (3) Words such as `which` (strong understood
  term) pull "which one runs first" (`t09`) to 0.082, a confused message called understood. The misses on `e10`, `h03` and `h08` all sit
  near 0.5.
- **The length-only baseline flags all 40 items as confused** (every probe item is shorter than nearly any training answer), so on the
  probe it *is* the constant predictor: accuracy 0.75, specificity 0.
- **Do not read the per-category recall (0.9 / 0.9 / 0.8) as detection.** It is what a model that answers "confused" to short or unfamiliar text
  would produce, and the near-chance AUC and 0.20 specificity show it is not telling the categories apart from understanding.

**What this means for the system.** The correctness proxy does not carry over to what students actually type: it cannot recognise explicit
understanding (2 of 10) and its confusion calls on the other 30 are mostly a bias, not a reading of the message. This is the evidence for
the rule layer, which has to handle **both** directions: explicit confusion phrases ("I don't understand", "I'm lost", "no idea") and
explicit understanding phrases ("got it", "that makes sense"). It also says the lexical model should not be the only decision-maker. With 40
items the intervals are wide, so the probe is a diagnostic of the gap, not a benchmark.

**Rule-based and ML-based classification together (IRWA Lecture 06; ARCHITECTURE ADR-04).** Lecture 06 contrasts the two families of text classifier. **Rule-based** classifiers use hand-written linguistic rules: they are precise and understandable for the phrasings they cover, but need expertise to write and are hard to maintain and scale. **ML-based** classifiers learn the mapping from labelled data and generalise, but only within the distribution they were trained on. This probe measures exactly that limit: trained on confident answers, the model cannot read "that makes sense" (P = 0.992 confused) and has no learned evidence for "I'm lost" (intercept only).

So the tutor service uses **both** (ADR-04): an explicit-phrase rule layer is checked **first**, in both directions (confusion and understanding), and the TF-IDF + LR model decides only when no rule matches. The service reports which one decided (`source = rule | model`) on every turn. That way the model is used where it has evidence (answer-like text) and the rules cover the explicit statements it demonstrably cannot handle. The rule list ships with the model as `rules.yaml` in the `clf-v1.0.0` artifact.

**The rules are deliberately not scored on this probe.** Their phrase list will be seeded from these 40 items, so evaluating the rules here would be circular: it would measure how well the list was copied, not how well it generalises. The probe's role is to justify that the rule layer exists. Measuring the rule layer itself needs new, unseen utterances, ideally real student messages in PP2.

## Findings of notebook 05

| Topic | Finding | Where |
|---|---|---|
| Reproduction | The refitted final model reproduces notebook 04's saved test rows exactly (TF-IDF + LR F1(confused) 0.7581, macro F1 0.6629; length-only 0.7651 / 0.6214). Same test set (fingerprint 511fadfe...). | Setup |
| Top features | The 40 top terms: 16 plausibly semantic (8 wrongness such as `only`, `but`, `error`; 8 correctness such as `implementation`, `declared`, `typically`), 8 length-proxy function words (all toward understood), 12 topic terms, 4 artefacts (`program`, `the program`, `doesn`, `that is`). | §10.2 |
| Leakage item 6 | **Mostly semantic or structural, not a leak.** But the lexical signal is generator style as much as correctness, and the rare topic terms rest on 2 to 7 answers. | §10.2 |
| Stability | 24 of 40 top terms stay in the top 20 in 4-5 of 5 train-only fold models, 14 in 2-3, **2 in 0-1 (noise)**. Frequent terms are stable; rare topic terms are not and should not be quoted as findings. | §10.3 |
| Length | corr(P(confused), token length) = **-0.154**: the model's score barely tracks length. TF-IDF + LR and length-only LR **disagree on 83 of 216 test rows** (both right 106, only model 43, only length 40, both wrong 27), so similar scores come from different signals. Combining them is a PP2 question. | §10.4 |
| U+2019 | 0 of 8,548 vocabulary terms are non-ASCII: the model cannot use the curly-apostrophe artefact. | §10.5 |
| **OOD probe** | The model flags **34 of 40** probe items as confused: recall 26/30 = 0.867 but **specificity 2/10 = 0.20**. Accuracy 0.70 is below "always confused" (0.75), balanced accuracy 0.533, ROC AUC **0.573** (near chance). Length-only flags all 40. | §11 |
| Proxy gap | The correctness proxy does not carry over to student phrasing. The model defaults to "confused" for short or unfamiliar text and mislabels "that makes sense" (P = 0.992) through the generator-style term `makes`. | §11 |
| Consequence | A rule layer is needed in **both** directions (explicit confusion and explicit understanding), and the lexical model should not decide alone. | §11 |
| Not shown | That confusion detection works. 40 probe items give wide intervals; the probe diagnoses the gap and is not a benchmark. | §11 |